# 01 – Data audit

Checks the Roboflow pork-rasher dataset before any training:

1. class balance per split,
2. photos that appear in more than one split (leakage),
3. a reproducible manual review of images with no defect labels.

Run in Google Colab. Add your Roboflow **private** API key to Colab Secrets as `ROBOFLOW_API_KEY` (key icon on the left) first.

In [ ]:
# Get the code (public repo, no token needed) and install the requirements.
# Safe to re-run: pulls the latest code if the repo is already cloned.
import os, sys

REPO = "/content/MMA3001-Project-repository"
if not os.path.isdir(REPO):
    !git clone https://github.com/rhal0027/MMA3001-Project-repository.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
!pip install -q -r requirements.txt

# Make the package importable straight away (an editable pip install
# only takes effect in Colab after a runtime restart).
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")


In [ ]:
# Download the dataset into data/raw/ (git-ignored)
from google.colab import userdata
from rasher_inspection import dataset as ds
from rasher_inspection import visualise as vis

root = ds.download_dataset(userdata.get("ROBOFLOW_API_KEY"), "data/raw/pork-rasher", version=4)
names = ds.load_class_names(root)
names

## 1. Class balance

In [ ]:
import pandas as pd

summary = ds.split_summary(root)
counts = pd.DataFrame({s: v["boxes"] for s, v in summary.items()})
info = pd.DataFrame({s: {k: v[k] for k in ("images", "sources", "empty_label", "missing_label")}
                     for s, v in summary.items()})
display(info)
display(counts)
counts.to_csv("results/class_counts.csv")
info.to_csv("results/split_info.csv")

## 2. Cross-split overlaps

Matched by Roboflow source id, so check each pair by eye: two different photos can share a file name.

In [ ]:
overlaps = ds.find_cross_split_overlaps(root)
print(len(overlaps), "source photos appear in more than one split")
vis.show_pairs(overlaps, class_names=names);

After looking at the pairs, list any that are **not** the same photo, then move the confirmed duplicates out of `train` (valid/test stay untouched). Run once with `dry_run=True` to check, then `False`.

In [ ]:
not_same_photo = []   # e.g. ["IMG_0012_jpg"] if a pair turned out to be different trays
confirmed = [o for o in overlaps if o["source"] not in not_same_photo]

moved = ds.quarantine_overlaps(root, confirmed, "data/quarantine", from_split="train",
                               log_csv="results/leakage_removed.csv", dry_run=True)
print(len(moved), "train images would be moved")

In [ ]:
# Re-check: should print 0 overlaps involving train
print([o["splits"] for o in ds.find_cross_split_overlaps(root)])

## 3. Manual review of unlabelled images

Write the criteria down **before** reviewing, e.g.:

- **clean** – no visible defect of any of the five classes
- **missed_defect** – a defect is visible but has no box (note which class)
- **unclear** – glare, blur or framing makes it impossible to judge

Record a verdict for each numbered image in the last cell, then download and commit the CSV.

In [ ]:
SPLIT, N, SEED = "test", 24, 0

unl = ds.find_unlabelled(root, SPLIT)
print({k: len(v) for k, v in unl.items()})
sample = ds.sample_for_review(unl["empty_label"], N, seed=SEED)
sheet = ds.write_review_sheet(sample, f"results/unlabelled_review_{SPLIT}_seed{SEED}.csv")
vis.show_grid(sample, cols=6, title=f"Unlabelled {SPLIT} images (seed {SEED})");
print("Fill in:", sheet)

In [ ]:
# Record a verdict for each image number shown above: "clean", "missed_defect" or "unclear"
verdicts = {
    # 0: "clean",
    # 1: "missed_defect",   # add a note below if useful
}
notes = {
    # 1: "loose meat bottom-left, not boxed",
}

review = pd.read_csv(sheet)
review["verdict"] = review["index"].map(verdicts)
review["notes"] = review["index"].map(notes)
review.to_csv(sheet, index=False)
print(review["verdict"].value_counts(dropna=False))
# Download results/*.csv from the Colab file browser and commit them from your computer.
